# 01_gold_views_and_catalog_inspection

## Purpose

Create analyst-facing Gold views and inspect the governed Unity Catalog structure.

This notebook prepares reporting views and confirms that Gold outputs are organized in the expected catalog and schema.

## Expected outputs

- `vattenfall_dev.analytics.vw_regional_operations_dashboard`
- `vattenfall_dev.analytics.vw_daily_incident_trends`
- `vattenfall_dev.analytics.vw_market_weather_overview`

## Expected governance checks

- catalog exists
- raw, refined, and analytics schemas are meaningful
- Bronze, Silver, and Gold tables are in the correct layers
- reporting views are available in the analytics schema

## Main idea

Gold views provide analyst-facing access patterns without duplicating unnecessary physical data.

In [0]:
catalog = "vattenfall_dev"
schema = "analytics"

dashboard_table = f"{catalog}.{schema}.gold_regional_operations_dashboard"
incident_table = f"{catalog}.{schema}.gold_grid_incident_summary"
market_table = f"{catalog}.{schema}.gold_daily_market_summary"
weather_table = f"{catalog}.{schema}.gold_weather_impact_summary"

dashboard_view = f"{catalog}.{schema}.vw_regional_operations_dashboard"
incident_view = f"{catalog}.{schema}.vw_daily_incident_trends"
market_weather_view = f"{catalog}.{schema}.vw_market_weather_overview"

print("Gold schema:", f"{catalog}.{schema}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {dashboard_view} AS
SELECT *
FROM {dashboard_table}
""")

print("Created view:", dashboard_view)

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {incident_view} AS
SELECT
    event_day,
    region,
    severity_band,
    incident_count,
    elevated_incident_count,
    total_duration_minutes,
    avg_duration_minutes,
    affected_asset_count,
    operational_status
FROM {incident_table}
""")

print("Created view:", incident_view)

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {market_weather_view} AS
SELECT
    m.report_day,
    m.region,
    m.avg_price_eur_mwh,
    m.max_price_eur_mwh,
    m.total_volume_mwh,
    m.is_high_market_price,
    w.avg_temperature_c,
    w.avg_wind_speed_kmh,
    w.max_wind_speed_kmh,
    w.total_precipitation_mm,
    w.weather_alert_count,
    w.weather_risk_signal
FROM {market_table} m
LEFT JOIN {weather_table} w
    ON m.report_day = w.report_day
    AND m.region = w.region
""")

print("Created view:", market_weather_view)

In [0]:
print("Gold tables and views in analytics schema:")

display(spark.sql(f"SHOW TABLES IN {catalog}.{schema}"))

In [0]:
print("Regional operations dashboard view:")
display(spark.table(dashboard_view).limit(20))

print("Daily incident trends view:")
display(spark.table(incident_view).limit(20))

print("Market weather overview view:")
display(spark.table(market_weather_view).limit(20))

In [0]:
print("Governance summary")

print("Catalog:", catalog)
print("Raw schema: Bronze ingestion and source-preserving tables")
print("Refined schema: cleaned and reusable Silver tables")
print("Analytics schema: Gold tables and reporting views")
print("Views created for analyst-facing and dashboard-facing consumption")
print("Catalog Explorer can be used to inspect the governed structure.")